In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/q10_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

# Project only necessary columns to reduce data transfer
orders_sub = orders[['O_ORDERKEY', 'O_CUSTKEY', 'O_ORDERDATE']]
lineitem_sub = lineitem[['L_ORDERKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_RETURNFLAG']]
customer_sub = customer[['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'C_NATIONKEY', 'C_ADDRESS', 'C_COMMENT']]
nation_sub = nation[['N_NATIONKEY', 'N_NAME']]

# Define date bounds
date1 = pd.Timestamp("1994-11-01")
date2 = pd.Timestamp("1995-02-01")

# GPU-native filtering using between and bracket indexing
osel = orders_sub['O_ORDERDATE'].between(date1, date2, inclusive='left')
lsel = lineitem_sub['L_RETURNFLAG'] == 'R'
forders = orders_sub[osel]
flineitem = lineitem_sub[lsel]

# Perform joins
jn = flineitem.merge(forders, left_on='L_ORDERKEY', right_on='O_ORDERKEY')
jn = jn.merge(customer_sub, left_on='O_CUSTKEY', right_on='C_CUSTKEY')
jn = jn.merge(nation_sub, left_on='C_NATIONKEY', right_on='N_NATIONKEY')

# Compute revenue per line
jn['TMP'] = jn['L_EXTENDEDPRICE'] * (1.0 - jn['L_DISCOUNT'])

# Aggregate and sort
gb = jn.groupby(
    ['C_CUSTKEY','C_NAME','C_ACCTBAL','C_PHONE','N_NAME','C_ADDRESS','C_COMMENT'],
    as_index=False,
    sort=False
)['TMP'].sum()
total = gb.sort_values('TMP', ascending=False)